# Práctica para evaluación módulo 'Large Language Models'
## RAG multi-turno y function calling con LangGraph

### Carga de API_KEY desde archivo de claves
Se cargan las API keys proporcionadas por el profesor (`GOOGLE_API_KEY` y `OPENAI_API_KEY`) de archivo .env en la raíz del proyect 

In [71]:
# Carga de variables de entorno (carga variables de /.env: GOOGLE_API_KEY y OPENAI_API_KEY)
import os
from load_dotenv import load_dotenv
load_dotenv()

True

### Elección de modelos
Dado que se plantea un RAG, se necesita un modelo para generación y otro para la generación de los embeddings que contendrán en contexto derivado de la documentación aportada.
Dado que no se requieren grandes capaciades y que se está en un entorno con un capacidad de uso de tokens limitada, se escogen:
- `GENERATION_MODEL`:
    **gemini-2.5-flash-lite**: *Our most cost-efficient multimodal model, offering the fastest performance for high-frequency, lightweight tasks. Gemini 2.5 Flash-Lite is best for high-volume classification, simple data extraction, and extremely low-latency applications where budget and speed are the primary constraints.*
- `EMBEDDING_MODEL`:
    **all-MiniLM-L6-v2**: Ahora se usa `ChromaDB` como base de datos vectorial. Se probó usando gemini-embedding-001, pero daba algunos errores y por simplicidad se ha usado el que viene por defecto   

In [72]:
# Elección modelos
# gemini-2.5-flash-lite:
# Our most cost-efficient multimodal model, offering the fastest performance for high-frequency, lightweight tasks. Gemini 2.5 Flash-Lite is best for high-volume classification, simple data extraction, and extremely low-latency applications where budget and speed are the primary constraints.
GENERATION_MODEL = 'gemini-2.5-flash-lite'


#EMBEDDING_MODEL = 'gemini-embedding-001'
# gemini-embedding-001:
# A specialized engine for high-dimensional vector representation, providing efficient numerical mapping of text and images. The Gemini Embedding model is best for semantic search, document retrieval, and recommendation systems that require fast, scalable similarity calculations across large datasets.

### Construcción de la función con su esquema
con el decorador `@tool` en LangChain una definición de función se registra como tool

In [108]:
import requests
from langchain_core.tools import tool
from typing import Any

def geocode_location(localidad: str) -> dict[str, Any]:
    """Obtiene coordenadas aproximadas para una ubicación usando Open-Meteo Geocoding."""
    response = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": localidad, "count": 1, "language": "es", "format": "json"},
        timeout=10,
    )
    response.raise_for_status()
    payload = response.json()
    results = payload.get("results") or []
    if not results:
        raise ValueError(f"No se encontraron coordenadas para: {localidad}")

    first = results[0]
    return {
        "name": first.get("name"),
        "country": first.get("country"),
        "latitude": first["latitude"],
        "longitude": first["longitude"],
        "timezone": first.get("timezone"),
    }

@tool
def get_current_weather(localidad: str) -> dict[str, Any]:
    """Consulta el tiempo actual en una localidad."""
    place = geocode_location(localidad)

    response = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": place["latitude"],
            "longitude": place["longitude"],
            "current": "temperature_2m,relative_humidity_2m,wind_speed_10m",
            "temperature_unit": "celsius",
            "timezone": "auto",
        },
        timeout=10,
    )
    response.raise_for_status()
    payload = response.json()
    current = payload.get("current") or {}
    units_payload = payload.get("current_units") or {}

    return {
        "localidad": place,
        "hora_medicion": current.get("time"),
        "temperatura": current.get("temperature_2m"),
        "unidad_temperatura": units_payload.get("temperature_2m"),
        "humedad": current.get("relative_humidity_2m"),
        "unidad_humedad": units_payload.get("relative_humidity_2m"),
        "viento": current.get("wind_speed_10m"),
        "unidad_viento": units_payload.get("wind_speed_10m"),
        "provider": "Open-Meteo",
    }


weather_tool_schema = {
    "type": "function",
    "name": "get_current_weather",
    "description": "Consulta el tiempo actual para una ciudad o ubicación. Úsala cuando el usuario pregunte por meteorología actual.",
    "strict": True,
    "parameters": {
        "type": "object",
        "properties": {
            "localidad": {
                "type": "string",
                "description": "Ciudad o ubicación. Ejemplos: Madrid, Illescas, Bogotá, Paris.",
            },
        },
        "required": ["localidad"],
        "additionalProperties": False,
    },
}

#weather_tool = ToolSpec(schema=weather_tool_schema, function=get_current_weather)

### Instanciar modelos
Se escoge `LangChain` como 'framework' en vistas a usar LangGraph más adelante, por la propiedad que tiene de estructurar un chat (sistema multi-turno)

En `GENERATION_MODEL` se establecen los siguientes parámetros:
- `temperature`=**0.2**. Se elige un valor bajo porqué se prima la fiabilidad del contenido a la forma. Esto es una respuesta quizá menos elaborada en términos de lenguage natural pero con mayor fiabilidad en que el contenido se ajuste a lo esperado. El motivo es que en una fase inicial del proyecto y con un llm ligero se quiere poder descartar al máximo las posibles fuentes de error. No se especifian ni top-P ni top-K porqué son hasta cierto punto redundantes con la temperatura y así no generar posibles inconsistencias en lo que el modelo entienda
- `max_output_tokens`=**400**. Valor final. Inicialemente se se escogió 150 para no correr el riesgo de vaciar la cuota antes de finalizar el ejercicio. 
- `response_mime_type`: **'text/plain'**. El por defecto, texto plano. En este punto se pretende un chat con el usuario, no redirigir a ninguna herramienta

En `GENERATION_MODEL` se registra la tool `get_current_weather` con el método `bind_tools`

In [74]:
# Instanciar los modelos
from langchain.chat_models import init_chat_model
from langchain_google_genai import GoogleGenerativeAIEmbeddings

llm = init_chat_model(GENERATION_MODEL,
                      model_provider="google_genai",
                      temperature=0.2,         # baja temperatura, se incentivan respuestas deterministas
                      #top_p=0.3,               # solamente toma en cuentra la sigueinte palabara más probable: puede resultar en repetición palabaras pero se asgura al ausencia de fallos
                      max_output_tokens=400,    # por no quedarse sin crédito en la práctica
                      response_mime_type='text/plain')

# se ligan las tools. El modelo sabrá cuando requerir una función por la información en su docstring
llm = llm.bind_tools([get_current_weather])


### Cargar documentación con el contexto
De momento se carga el pdf, tal cual, con `PyPDFLoader`. El formato de texto absolutamente plano que se obtiene no permite hacer un chunking que aproveche la estructura que proveen los títulos. Se ve que el modelo no funciona muy bien con esto, aun probando distintos tamaños y overlap. Se deja para una fase posterior cargar el pdf previamiente convertido a markdown para solucionarlo

In [75]:
# Cargar el documento
from langchain_community.document_loaders import PyPDFLoader

pages = PyPDFLoader('data/TENERIFE.pdf').load()

In [76]:
pages[0].metadata

{'producer': 'PyPDF',
 'creator': 'Microsoft Word',
 'creationdate': '2025-07-13T20:00:01+00:00',
 'title': 'Microsoft Word - TENERIFE.docx',
 'moddate': '2025-07-13T20:00:01+00:00',
 'source': 'data/TENERIFE.pdf',
 'total_pages': 25,
 'page': 0,
 'page_label': '1'}

### Chunking
Se usa `RecursiveCharacterTextSplitter` que usa la puntuación para ajustar donde corta el chunk.
En el documento, se ha visto que, más o menos, el trozo mas largo de contenido semánticamente homogéneo es de unos 900 carácteres. También los hay mucho más cortos. Se han probado varias opciones en longitud y overlap, sin conseguir que ninguna sea capaz de hacer un retrive correcto a 'cuales son las playas de la Orotava'

Para poder tracear mejor el resultado se añade en metadata un identificador del chunk y su tamaño. El nombre del documento y la página de donde se ha extraído el chunk ya los pone por defecto PyPDFLoader en 'source' y 'page_label'

In [77]:
# Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=80,
    add_start_index=True,
)

chunks = text_splitter.split_documents(pages)

In [78]:
for i, chunk in enumerate(chunks):
    chunk.metadata['chunk_id']=i
    chunk.metadata['len']=len(chunk.page_content)

chunks[0].metadata

{'producer': 'PyPDF',
 'creator': 'Microsoft Word',
 'creationdate': '2025-07-13T20:00:01+00:00',
 'title': 'Microsoft Word - TENERIFE.docx',
 'moddate': '2025-07-13T20:00:01+00:00',
 'source': 'data/TENERIFE.pdf',
 'total_pages': 25,
 'page': 0,
 'page_label': '1',
 'start_index': 0,
 'chunk_id': 0,
 'len': 324}

### Creación de la base de datos vectorial
Se usa `ChromaDB`. Por simplicidad no se persiste en disco


In [81]:
import chromadb
from chromadb.utils import embedding_functions

#client = chromadb.PersistentClient(path=CHROMA_DATA_PATH)
#client.delete_collection('TenerifePOI')
client = chromadb.Client()  # no se persisten los datos, por simplicidad

embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
collection = client.create_collection(name='TenerifePOI',
                                      embedding_function=embedding_func,
                                      metadata={"hnsw:space": "cosine"})
collection.add(documents=[chunk.page_content for chunk in chunks],
               ids=[f"id{chunk.metadata['chunk_id']}" for chunk in chunks],
               metadatas=[chunk.metadata for chunk in chunks]
)


In [109]:
query_res = collection.query(query_texts=["playas en el Norte"],
                                 n_results=3)
query_res.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances'])

In [83]:
query_res['documents']

[['De esta zona os recomendaríamos las siguientes playas: \no Playa de Torviscas',
  'tendréis poca playa (especialmente a la del Ancón) y tened mucho respeto al mar en \nestas playas, que la corriente es muy traicionera. \n \n• Puerto de La Cruz \nEs la zona turística del Norte de Tenerife. De hecho, hace años era el núcleo turístico \nde la isla hasta que empezó a ganar mucho peso el Sur de Tenerife.',
  'o Playa de Benijo [vídeo - ubicación] \nPlaya de arena negra, si el día está despejado, de los atardeceres más \nespectaculares que podéis ver en la isla. \nPara llegar a esta playa, aparcar por aquí y, para acceder a la playa, se accede \ndesde aquí (puede ser que os encontréis con el acceso cortado con una valla, \npero da igual, podéis acceder igualmente). \n• La Orotava']]

### Definición del objeto RAGState
Contiene el estado del RAG:
- histórico de respuestas
- contexto para el último mensaje de usuario

In [110]:
from typing import Annotated, Sequence
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage

class RAGState(TypedDict):
    # messages es una secuencia (lista en sentido amplio) de mensajes LangChain (BaseMessage es clase base para HUmanMessage, SystemMessage, AIMessage y ToolMessage)
    # con el comentario 'add_messages' se indica que la función 'add_messages' de LangGraph añada la respuesta obtenida a la lista
    messages: Annotated[Sequence[BaseMessage], add_messages]
    # context se va a actualizar en cada iteración con el retrieval asociado al último mensaje de usuario
    context: str

### Creación del prompt template con LangChain 'placeholder' para que se actualice en cada turno con el histórico de respuestas y el mensaje de usuario y con control de uso de tokens

Con ello se puede encapsular de una vez las instrucciones de sistema y poder entrar como variables pregunta y contexto obtenido en la fase de 'retrieval'

En el mensaje de 'system':
- se especifica que únicamente se tenga en cuenta el contexto proporcionado por la documentación incluida en el RAG. No se quieren 'interferencias' externas, se quiere evaluar la capacidad de manejar la documentación dada
- se pide que si en el contexto no encuentra la información, que claramente lo diga, que no invente
- se pide que cite la fuente (se va a incluir un campo 'Fuente' en los metadatos de los chunk

Se cambia el tipo 'user' por 'placeholder', de forma que LangGraph va rellenar automáticamente con RAGState['messages'], que contiene el histórico de respuestas más el último mensaje de usuario en cada turno

El contexto se incluye como mensaje de usuario. El contexto se actualizará con cada nuevo mensaje de usuario en el nodo de retrieval en el grafo de LangGraph

control de tokens no funciona. el modelo elegido no es soportado por tiktoken. Se debería implementar algo parecido con métodos nativos de google. La idea era:  Para limitar el número de tokens de la consulta se usa `tiktoken` para ir sumando empezando por el final y cuando se supera el límite se van quitando los mensajes más antiguos del histórico. Más concretamente, se implementa la función `token_usage_review(state: RAGState, context: str, max_tokens: int = 200) -> TypeDict('cut': str, 'spare': int)`, que retrona la diferencia con el máximo: si es negativa es que faltan. Tamién en qué punto corta si el punto de corte está en el histórico recorta lo más antiguo para cumplir.

In [95]:
# Prompt template
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import MessagesPlaceholder
import tiktoken

system_prompt = '''Eres un asistente para dar información sobre lugares de interés en Tenerife
             Responde únicamente usando el contexto
             Para asuntos relacionados con el tiempo dispones de una herramienta
             Si el contexto no contiene la respuesta, di claramente: información no disponible en el contexto
             No inventes datos
             Cita Fuentes al final'''

make_prompt = ChatPromptTemplate.from_messages(
    [
        (
             'system', 
             system_prompt,
        ),
        MessagesPlaceholder("messages"),
        (
            'user',
            '{context}'
        )
    ]
)  

"""
tokens = gemini_client.models.count_tokens(
    model=GOOGLE_MODEL,
    contents=question,
)
"""

"""
enc = tiktoken.get_encoding(GENERATION_MODEL)

def token_usage_review(state: RAGState, context: str, max_tokens: int = 500):
    ''' 
    Retrona la diferencia con el máximo: si es negativa es que faltan. 
    Tamién en qué punto corta
    si el punto de corte está en el histórico recorta lo más antiguo para cumplir
    '''

    token_budged = max_tokens
    
    token_budget -= len(enc.encode(system_prompt))
    if token_budget < 0:
        return {'cut': 'system_prompt',
                'spare': token_budget}
    
    token_budget -= len(enc.encode(context))
    if token_budget < 0 :
        return {'cut': 'context',
                'spare': token_budget}
    
    initial_len = len(state['messages'])
    9
    for i in range(-1, -(initial_len +1) , -1) :
        token_budged -= len(enc.encode(state['messages'][i].content))
        if token_budged < 0 :
            i += 1
            state['messages'] = state['messages'][i:]
            break
    
    return {'cut': None if i == -1 else f'{initiual_len + i} older messages',
            'spare': token_budget}
"""


"\nenc = tiktoken.get_encoding(GENERATION_MODEL)\n\ndef token_usage_review(state: RAGState, context: str, max_tokens: int = 500):\n    ''' \n    Retrona la diferencia con el máximo: si es negativa es que faltan. \n    Tamién en qué punto corta\n    si el punto de corte está en el histórico recorta lo más antiguo para cumplir\n    '''\n\n    token_budged = max_tokens\n\n    token_budget -= len(enc.encode(system_prompt))\n    if token_budget < 0:\n        return {'cut': 'system_prompt',\n                'spare': token_budget}\n\n    token_budget -= len(enc.encode(context))\n    if token_budget < 0 :\n        return {'cut': 'context',\n                'spare': token_budget}\n\n    initial_len = len(state['messages'])\n    9\n    for i in range(-1, -(initial_len +1) , -1) :\n        token_budged -= len(enc.encode(state['messages'][i].content))\n        if token_budged < 0 :\n            i += 1\n            state['messages'] = state['messages'][i:]\n            break\n\n    return {'cut': N

### Generación del string de contexto a partir de los chunk obtenidos en 'retrieval'
El contexto debe pasarse como un string al `prompt template`. Se incluyen los metadatos que interessan (para cada chunk, documento fuente, página del documento e identificador de chunk) para poder auditar la respuesta

In [96]:
# Generación de string de contexto a partir de chunks devueltos por similarity_search
def make_context_string(query_res):
    context = ""
    for metadata, doc in zip(query_res['metadatas'][0], query_res['documents'][0]):
        context = context + f"Fuente: {metadata['source']},\
        Página: {metadata['page_label']},\
        chunk: {metadata['chunk_id']}\
        \n{doc}\n\n"
    return context
        

In [97]:
print(make_context_string(query_res))

Fuente: data/TENERIFE.pdf,        Página: 16,        chunk: 36        
De esta zona os recomendaríamos las siguientes playas: 
o Playa de Torviscas

Fuente: data/TENERIFE.pdf,        Página: 11,        chunk: 18        
tendréis poca playa (especialmente a la del Ancón) y tened mucho respeto al mar en 
estas playas, que la corriente es muy traicionera. 
 
• Puerto de La Cruz 
Es la zona turística del Norte de Tenerife. De hecho, hace años era el núcleo turístico 
de la isla hasta que empezó a ganar mucho peso el Sur de Tenerife.

Fuente: data/TENERIFE.pdf,        Página: 8,        chunk: 11        
o Playa de Benijo [vídeo - ubicación] 
Playa de arena negra, si el día está despejado, de los atardeceres más 
espectaculares que podéis ver en la isla. 
Para llegar a esta playa, aparcar por aquí y, para acceder a la playa, se accede 
desde aquí (puede ser que os encontréis con el acceso cortado con una valla, 
pero da igual, podéis acceder igualmente). 
• La Orotava




### Definición de nodo retrieval
Este nodo va a actualizar RAGState['context'] con el retrieval de la base de datos vectorial

In [111]:
k = 3  # k deja de ser un parámetro 'de usuario' para ser una constante de configuración
def retrieve_node(state: RAGState):
    
    # el último mensaje de RAGState['messages'] es la entrada de usuario, los anteriores el histórico de respuestas
    query_res = collection.query(query_texts=[state['messages'][-1].content], n_results=k)
    
    # retorna el string de contexto en diccionario con la etiqueta 'context', lo que permite a LangChain ligar con RAGState['conext']
    return {'context': make_context_string(query_res)}

### Definición de nodo de generación con tool calling
Crea el prompt a partir RAGState actulizado, lo entra en el llm, si este detecta qeu se necesita tool, se ejecuta, se actualiza RAGState con la respuesta y se llama de nuevo al llm para generar la respuesta, que al retornar va a ser incluida por el sistema a RAGState. En cada paso intrerno se actualizan los mensajes generados en RAGState.

In [112]:
from langchain_core.messages import ToolMessage

def generate_node(state: RAGState):
    
    # se crea el prompt
    prompt = make_prompt.invoke({'messages': state['messages'], 'context': state['context']})
    
    # se llama al llm
    ai_message = llm.invoke(prompt)

    # se evalúa si la respuesta pide la ejecuciómn de una tool
    if ai_message.tool_calls:
        # se guarda el mensaje en el RAGState
        state['messages'].append(ai_message)

        # se itera sobre tolas las tool_call que el modelo ha estimado necesita
        for tool_call in ai_message.tool_calls:
            tool_name = tool_call['name']
            tool_args = tool_call['args']
            
            # para cada tool se mira si es requerida
            if tool_name == 'get_current_weather':
                # en caso de ser la tool requerida se ejecuta y se guarda su resultado en RAGState como un tool_message
                try:
                    tool_res = get_current_weather.invoke(tool_args['localidad'])
                except Exception as e:
                    tool_res=f'ERROR in {tool_name} with {str(tool_args)}: {e}' 
                {'localidad': {'name': 'Santa Cruz de Tenerife', 'country': 'España', 'latitude': 28.46824, 'longitude': -16.25462, 'timezone': 'Atlantic/Canary'}, 'hora_medición': '2026-06-17T18:30', 'temperatura': 22.7, 'unidad_temperatura': '°C', 'humedad': 64, 'unidad_humedad': '%', 'viento': 12.2, 'unidad_viento': 'km/h', 'provider': 'Open-Meteo'}
                tool_message =ToolMessage(content= \
                                          f"tiempo en {tool_res['localidad']['name']}: " \
                                          f" a las {tool_res['hora_medición']}, " \
                                          f"la temperatura es de {tool_res['temperatura']}{tool_res['unidad_temperatura']}",
                                          tool_call_id=tool_call['id'],
                                          name=tool_call['name'])
                state['messages'].append(tool_message)
        
        # usa vez ejecutadas las tools y guardado su resultado en RAGState, se calcula de nueveo el promt y se lanza query al modelo
        prompt = make_prompt.invoke({'messages': state['messages'], 'context': state['context']})
        ai_message = llm.invoke(prompt)
        print(ai_message)

    # se retorna la respuesta en diccionario con la etiqueta 'messages', lo que permite a LangChain linkar con RAGState['messages']
    return {'messages': ai_message}

### Construcción del grafo con memoria

In [100]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END

# Definición del objeto grafo a partir de RAGState
workflow = StateGraph(RAGState)

# Creación de los objetos nodos del grafo
workflow.add_node('retriever', retrieve_node)
workflow.add_node('generator', generate_node)

# Ubicación de los nodos en el grafo (princpio, final y conexiones)
workflow.add_edge(START, 'retriever')
workflow.add_edge('retriever', 'generator')
workflow.add_edge('generator', END)

# Compilación del grafo con la memoria, gestionada en RAGState por InMemorySaver
graph = workflow.compile(checkpointer=InMemorySaver())

### Constucción de la función de query
Tiene como parámetros la prenguta de usuario y thread_id. thread_id es un hilo de conversación; si se quiere seguir con esa conversación se debe introducir su thread_id. k es ahora un parámetro de sistema, seteado en el nodo de retrieval
Con cada llamada imprime todo el histórico de mensajes. Hay la opción de que incluya el printado del contexto, a efectos de auditoria (deshabilitado por defecto)
Ejecuta la llamada al grafo (workflow LangGraph)

In [101]:
from langchain_core.messages import HumanMessage

def TenerifePOI(question: str, chat_id: str, show_context=False):
    
    res = graph.invoke({'messages': HumanMessage(content=question)},
                 config={'configurable': {'thread_id': chat_id}})

    for m in res['messages']:
        message_type = str(type(m)).split("'")[1].split(".")[-1]
        message_content = m.content
        print(f'--- {message_type} ---:\n {message_content}')
        print(80*"=")

    if show_context:
        print('\n--- CONTEXT ---')
        print(res['context'])
        print(80*"=")

    return res

### Prueba de la query
Falla en la pregunta 'playas en la Orotava'. Son Bollullo, Patos y Ancona. Se piensa que esto se podría mejorar si el chunking tomase los títulos: la información tiene un sentido, de título en adelante, no hacia los dos lados; Las 3 playas están entre dos títulos, conteniendo el primero 'Orotava'

In [106]:
res = TenerifePOI('dime el mejor restaurante en Santa Cruz de Tenerife', 'Sta Cruz ii')

--- HumanMessage ---:
 tiempo actual Santa Cruz de Tenerife
--- AIMessage ---:
 
--- ToolMessage ---:
 tiempo en Santa Cruz de Tenerife:  a las 2026-06-17T21:15, la temperatura es de 20.1°C
--- AIMessage ---:
 El tiempo actual en Santa Cruz de Tenerife es de 20.1°C.

Fuente: data/TENERIFE.pdf
--- HumanMessage ---:
 qué playas hay en Santa Cruz de Tenerife
--- AIMessage ---:
 En Santa Cruz de Tenerife se encuentran las playas del Ancón.

Fuente: data/TENERIFE.pdf
--- HumanMessage ---:
 donde pudo comprar una excavadora en Santa Cruz de Tenerife
--- AIMessage ---:
 Información no disponible en el contexto
--- HumanMessage ---:
 dime el mejor restaurante en Santa Cruz de Tenerife
--- AIMessage ---:
 El Brunelli's Steakhouse se encuentra cerca del Loro Parque en el Puerto de la Cruz. Los Guachinches son típicos de la zona norte de Tenerife.

Fuente: data/TENERIFE.pdf


In [107]:
res = TenerifePOI('dime las playas de la Orotava', 'Orotava iii')

--- HumanMessage ---:
 dime las playas de la Orotava
--- AIMessage ---:
 A La Orotava pertenecen tres playas: La Playa del Bollullo, Playa de Torviscas y Playa de Benijo.
